Welcome to the module introducing `pandas` in AccFin Research.

`pandas` is the most useful Python library for data analysis in our research. In the second module, you will learn how to use `pandas` to handle data in tabular form. We will cover the fundamental operations needed to load data, explore it, clean it, and prepare it for further analysis. By the end, you’ll have the skills to manage real-world datasets and apply `pandas` to replicate a saminal paper.

**Learning Outcomes**

By completing this tutorial, you will be able to:

- Create and explore *Series* and *DataFrames*, the two core data structures in `pandas`.

- Import and export datasets from common formats such as CSV and Stata.

- Inspect and summarize data using built-in functions.

- Select, filter, and slice data to focus on what matters.

- Clean and transform datasets by handling missing values, renaming columns, and changing data types.

- Perform basic data analysis with sorting, grouping, and simple aggregations.

- Integrate pandas workflows into larger Python projects.

In [ ]:
import numpy as np
import pandas as pd

### 1. pandas Series

In [ ]:
#create a Series object from a Python List object
List = ['Tesla','2023','1 TESLA ROAD, AUSTIN, TX', 106_618]
series = pd.Series(List)
print(series)

In [ ]:
#create a Series object from a Numpy array object
array1 = np.array(['Tesla','2023','1 TESLA ROAD, AUSTIN, TX', 106_618])
series = pd.Series(array1)
print(series)

In [ ]:
#Slicing a Series object
print(series[0])
print(series[3:])

In [ ]:
#create a Series object, dict1, from a dictionary object
dict1 = {"CompName":"Tesla", "fyear":2023, "Address": "1 TESLA ROAD, AUSTIN, TX", "at": 106_618}
series1 = pd.Series(dict1)
print(series1)

In [ ]:
series2 = pd.Series(['Tesla',2023,'1 TESLA ROAD, AUSTIN, TX', 106_618], index=['CompName', 'fyear', 'Address', 'at'])
print(series2)

In [ ]:
#Slicing a Series object by the Index
print(series1["fyear"])
print(series1["Address":])

The difference between a `list` and a pandas Series object is that:

- The elements in a Series object can be indexed by a label.
- A Series object can directly apply comparison:

In [ ]:
List = [1,2,3,4,5,6,7,8,9,10]
series = pd.Series(List)

print(series > 5)
print('However, List cannot be compared to a scalar value:')

try:
    print(List > 5) #This will throw an error
except Exception as e:
    print(f'Error: {e}')

In [ ]:
series[series > 5]

In [ ]:
series1 == series2

### 2. pandas DataFrame - Basic
The difference between a *Series* object and a *DataFrame* object is that a Series object has only one row, while a DataFrame object is a table with multiple rows.

#### 2.1. Load data

In [ ]:
#From a Numpy matrix (rarely used)
np.random.seed(0)
matrix = np.random.randn(4,5)
df0 = pd.DataFrame(matrix, ['F','GM','RACE','TSLA'], ['sale','ni','at','lt', 'seq'])
print(df0)

In [ ]:
#Load data from a csv file
df = pd.read_csv('data/comp_sample.csv', parse_dates=['datadate'])

# pd.read_stata('comp_sample.dta'); 
# pd.read_excel('comp_sample.xlsx'); 
# pd.read_sas('comp_sample.sas7bdat', format = 'sas7bdat', encoding="utf-8")

# df.to_csv('comp_sample.csv', index=False)
# df.to_stata('comp_sample.dta', ignore_index=True)

#### 2.2. DataFrame Attributes

In [ ]:
#get a list of columns using the `columns` attrniute
df.columns

In [ ]:
list(df.columns)

In [ ]:
df.dtypes

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df['at'].dtype

In [ ]:
#Rename columns
df.rename(columns={'conm':'Company Name'}, inplace=True)

#### 2.3. Accessing DataFrames

In [ ]:
df.head()
#df.head(3)
#df.tail()
#df.sample(20)

In [ ]:
# Slicing a DataFrame object
df[3:10]

##### `.iloc` and `.loc`
**Key Differences**
|Feature|.loc|.iloc|
| --- | --- | --- |
|Selection|Label-based|Integer position-based|
|Slicing|Inclusive|Exclusive (Python's standard slicing)|
|Error Handling|Raises `KeyError` for invalid label|Raises `IndexError` for invalid position|	
		
		
		

In [ ]:
# Index location method
df.iloc[3:10, :2]

In [ ]:
df.iloc[:2]

In [ ]:
# Label location method
df.loc[0:10, ['gvkey','ni']]

##### Selecting columns

In [ ]:
df['gvkey']
# Equivalent to `df.gvkey` or `df.loc[:, 'gvkey']`

In [ ]:
df[['gvkey', 'fyear', 'ni']]

##### Drop columns using `.drop()`

In [ ]:
df1 = df.drop(columns = ['ni','oancf'])

In [ ]:
# Equivalent to
df1 = df.drop(['ni','oancf'], axis=1)

#### 2.4. Filtering
(Selecting Rows)

In [ ]:
df['ni'] < 0

In [ ]:
df[df.ni < 0]

In [ ]:
#Combining rows and columns selection
df[df['ni']<0][['gvkey', 'fyear', 'ni']]

In [ ]:
df.loc[df['ni'] < 0]

In [ ]:
df.loc[df['ni'] < 0, ['gvkey','fyear','ni']]

In [ ]:
df[df['gvkey']==1690]
# Equivalent to df.loc[df.gvkey==1690]

In [ ]:
#Filtering string variables
df[df['Company Name'].str.contains('APPLE')].head()

In [ ]:
# .isin() creates a boolean series whenter a condition is met.
df[df['gvkey'].isin([1690,12141])]

In [ ]:
cond1 = df['gvkey'].isin([1690,12141])
cond2 = df['fyear']==2022
df[cond1 & cond2]

In [ ]:
#use `&` and `|` for multiple conditions
df1 = df[( (df['ni'] < 0) | (df['ni'] > 5_000) ) & (df['at']>1000) & (df['au'].isin([1,2,3,4]))]
df1.head()

##### 2.4.1. query()

The `df.query()` method in pandas is a versatile tool for filtering DataFrame rows using string expressions. It supports a wide range of operations, allowing you to create complex queries that might be cumbersome with traditional boolean indexing. Here are some of the filtering capabilities you can achieve with `df.query()`:

- 1. **Basic Comparisons**

You can use comparison operators like `>`, `<`, `>=`, `<=`, `==`, and `!=` within the query string to filter data based on numeric or date-time conditions.

You can also check for membership `in` a list or array, which is useful for filtering categories or groups.

You can combine conditions using logical operators like `&`, `|`, and `~`.

In [ ]:
df.query('~((fyear != 2022) | ~(au in [1,2,3,4])) & (ni < 0)')

- 2. **String Operations**

`query()` supports string methods that can be used to filter rows based on string conditions. You must use the `str` accessor.

In [ ]:
#If the variable name contains spaces, use backticks
df.query('`Company Name`.str.startswith("A")')

- 3. **Null Checks**

You can check for null (or non-null) values using `isnull()` and `notnull()`.

In [ ]:
df.query('xrd.notnull()')

- 4. **Variable Substitution**

You can include external variables in your query by prefixing them with an `@` symbol. This is useful for dynamic queries based on variable values.

In [ ]:
threshold = 1000
df.query('at > @threshold')

In [ ]:
Big4 = [1,2,3,4]
df.query('au in @Big4')

- 5. **Complex Expressions**

You can use more complex expressions involving arithmetic operations, functions, and more.

In [ ]:
df.query('(ni / (prcc_f*csho)) > 0.1 & at < 50000')

**Note**: For some operations, especially those involving string methods or checking for null values, you need to specify the query engine as `python` because the default `numexpr` engine does not support all operations.

#### 2.5. Missing data

In [ ]:
# `isnull` creates a boolean series for whether an observation is missing; `isna()` also works here
df['xrd'].isnull()

In [ ]:
df.xrd.isnull().sum() / len(df)

In [ ]:
df1 = df[df['xrd'].isnull()]
df1.head()

In [ ]:
#`notnull` creates a boolean series for whether an observation is NOT  missing; `notna()` also works here
df2 = df[df['xrd'].notnull()]
df2.head()

In [ ]:
# `dropna()` drops observations with missing values
df3 = df.dropna(subset=['xrd'])

In [ ]:
df.dropna(subset=['gvkey', 'fyear', 'at'], inplace=True)

In [ ]:
# `.fillna()` method - backfills with specified value
df3 = df.copy()
df3['xrd'] = df3['xrd'].fillna(0)
df3

In [ ]:
df4 = df.copy()
df4['oancf'] = df4['oancf'].fillna(df['oancf'].mean())
df4

#### 2.6. Sort

In [ ]:
df.sort_values(by=['gvkey','fyear'], inplace=True)

In [ ]:
df.sort_values(by=['gvkey','fyear', 'datadate'], ascending=[False, False, False], inplace=True)

In [ ]:
#Reset the index
df.reset_index(inplace=True, drop=True)

#### 2.7. Duplicates

In [ ]:
df[['gvkey', 'fyear']].duplicated().sum()

keep='first' (default): Marks duplicates as True except for the first occurrence.

keep='last': Marks duplicates as True except for the last occurrence.

keep=False: Marks all duplicates as True.

In [ ]:
df[['gvkey', 'fyear']].duplicated(keep=False)

In [ ]:
df1 = df.drop_duplicates()

When removing duplicates, we have 3 options:
- `df.drop_duplicates(subset=['gvkey','fyear'], inplace=True)` keeps the **first** occurrence of unique combinations of `gvkey` and `fyear`.
- `df.drop_duplicates(subset=['gvkey','fyear'], keep='last', inplace=True)` keeps the **last** occurrence of unique combinations of `gvkey` and `fyear`.
- `df.drop_duplicates(subset=['gvkey','fyear'], keep=False, inplace=True)` keeps **none** of the duplicated combinations of `gvkey` and `fyear`.

In [ ]:
df2 = df.drop_duplicates(subset=['gvkey','fyear'], keep='last')

In [ ]:
# My practice of keeping unique gvkey-fyear pairs
Variables_used_in_test = ['ceq','ni', 'xrd', 'capx', 'oancf','prcc_f', 'csho']
df['Missing_Count'] = df[Variables_used_in_test].isna().sum(axis=1)

df = df.sort_values(by=['gvkey','fyear', 'Missing_Count', 'datadate'], ascending=[True, True, True, False])\
    .drop_duplicates(subset=['gvkey','fyear'])\
    .drop(columns=['Missing_Count']).reset_index(drop=True)

#### 2.8. Generate Variables

In [ ]:
df['MV'] = df['prcc_f'] * df['csho']

In [ ]:
df['Size'] = np.log(df['at']).round(5)

In [ ]:
#Change data type
df['fyear'] = df['fyear'].astype('int')

In [ ]:
# Useful: Generate Xtiles
df['Size_tile'] = pd.qcut(df['at'], q=10, labels=[i for i in range(1,11)])

##### 2.8.1. Working with Date

In [ ]:
# Turn a string or interger object into a datetime object
# If you have not included the `parse_dates = ['datadate']` argument in the `pd.read_csv()` function
df['datadate'] = pd.to_datetime(df['datadate'])

**Date Format**

`%Y`: Year with century as a decimal number.

`%y`: Year without century as a zero-padded decimal number (00-99).

`%m`: Month as a zero-padded decimal number (01-12).

`%B`: Full month name (January - December).

`%b`: Locale’s abbreviated month name (Jan - Dec).

`%d`: Day of the month as a zero-padded decimal number (01-31).

`%H`: Hour (24-hour clock) as a zero-padded decimal number (00-23).

`%I`: Hour (12-hour clock) as a zero-padded decimal number (01-12).

`%M`: Minute as a zero-padded decimal number (00-59).

`%S`: Second as a zero-padded decimal number (00-59).

`%p`: Locale’s equivalent of either AM or PM.

`%A`: Locale’s full weekday name.

`%a`: Locale’s abbreviated weekday name.

`%c`: Locale’s appropriate date and time representation.

In [ ]:
df['datadate1'] = df['datadate'].dt.strftime('%d-%b-%Y')

In [ ]:
df['year'] = df['datadate'].dt.year
df['qtr'] = df['datadate'].dt.quarter

##### 2.8.2. apply() function

In [ ]:
df['Size'] = df['at'].apply(lambda x: np.log(x).round(5))

In [ ]:
df['RnD_NotNA'] = df['xrd'].apply(lambda x: 0 if x is None else x)

In [ ]:
df['Value'] = df.apply(lambda x: 1 if x['ceq'] > x['MV'] else 0, axis=1)

In [ ]:
def plusone(x: int) -> int:
    return x + 1

df['year_plus_one'] = df['fyear'].apply(plusone)
# df['year_plus_one'] = df['fyear'].apply(lambda x: x + 1)

### 3. Summary statistics

In [ ]:
#Number of non-missing observations
df.count()

In [ ]:
#Frequency of each unique value
df['Company Name'].value_counts()

In [ ]:
# Number of Unique values
df['gvkey'].nunique()

In [ ]:
df['at'].sum()

In [ ]:
df['at'].std()

In [ ]:
df['at'].quantile(0.99)

In [ ]:
df["at"].quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])

In [ ]:
df[['at','sale','oancf']].describe()

In [ ]:
# Summary statistics
df[['at','sale','oancf']].describe([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])

In [ ]:
# Correlation matrix
df[['at','sale','oancf']].corr(method='pearson')

In [ ]:
df[['at', 'ceq']].aggregate(['count','mean','median','std', 'var', 'min','max','sum','quantile','skew', 'kurt'])

#### 3.1. Winsorization and Trucation

In [ ]:
#Winsorize Asset:
df['at_w'] = df['at'].clip(lower = df['at'].quantile(0.01), upper = df['at'].quantile(0.99))

#Truncate:
df['at_t'] = df['at'].mask((df['at'] < df['at'].quantile(0.01)) | (df['at'] > df['at'].quantile(0.99)))

In [ ]:
def winsor2(x: pd.Series, p: float = 0.01, truncate: bool = False) -> pd.Series:
    if p > 0 and p < 0.5: 
        if truncate:
            return x.mask((x < x.quantile(p)) | (x > x.quantile(1-p)))
        else:
            return x.clip(lower = x.quantile(p), upper = x.quantile(1-p))
    else:
        raise ValueError('p must be between 0 and 0.5')

In [ ]:
df['at_w'] = winsor2(df['at'])
df['at_t'] = winsor2(df['at'], p=0.05, truncate=True)

In [ ]:
df[["at", 'at_w', 'at_t']].describe([0.01,0.05,0.25, 0.5, 0.75, 0.95, 0.99])

In [ ]:
# Alternatively, you can use the `winsorize` function from the `scipy.stats.mstats` module
from scipy.stats.mstats import winsorize
df['at_w1'] = winsorize(df['at'], limits=[.01,.01], inclusive=[False, False])

### 4. Groupby

#### 4.1 Summarizing: `.describe()`
Using describe() on a groupby object to get a quick overview of the statistics for each group is also very common.

In [ ]:
# Generate descriptive statistics for each group
df.groupby('fyear')['at'].describe()

You can also calculate the mean, median, sum, min, and max for each group.
- `count()`, `.sum()`	Total number of items
- `first()`, `last()`	First and last item
- `mean()`, `median()`	Mean and median
- `min()`, `max()`	Minimum and maximum
- `std()`, `var()`	Standard deviation and variance
- `prod()`	Product of all items
- `sum()`	Sum of all items

In [ ]:
# Count non-NA cells for at in each year
df.groupby('fyear')['at'].count()

In [ ]:
df.groupby('fyear')['at'].median()

In [ ]:
df.groupby('fyear')[['at','ni','oancf']].max()

In [ ]:
df1 = df.groupby('fyear')[['at','ni','oancf']].mean().reset_index()
df1.head()

In [ ]:
df2 = df.groupby('fyear', as_index=False)['at'].mean()
df2.head()

#### 4.2. Aggregation: `.agg()`
Aggregation is one of the most common uses of groupby(), where you compute a summary statistic (or statistics) about each group. 

It returns **a scalar** per group.

In [ ]:
df3 = df.groupby('fyear', as_index=False).agg({'at': ['mean','median','std'], 
                                             'ni': ['sum','mean','count'],
                                             'prcc_f': ['min','max']})
df3

In [ ]:
#Turn the variable names into a single level
df3.columns = ['_'.join(col).strip() for col in df3.columns.values]

#### 4.3. Transformation: `.transform()`
Transformation returns **a DataFrame that is the same size as the input** and is useful for operations such as filling NAs within groups with a value derived from each group.

Similar to `egen var_new = fun(var_old), by(group)` in Stata

In [ ]:
df['mean_at'] = df.groupby('fyear')['at'].transform(lambda x: x.mean())

In [ ]:
# Standardize data within each group
df['standardized_at'] = df.groupby('fyear')['at'].transform(lambda x: (x - x.mean()) / x.std())

In [ ]:
# Rank data within each group
df['at_port'] = df.groupby('fyear')['at'].transform(lambda x: pd.qcut(x, 10, labels=False)) + 1

In [ ]:
# Fill NA values within each group with the group’s mean
df['xrd_filled_na'] = df.groupby('fyear')['xrd'].transform(lambda x: x.fillna(x.mean()))

In [ ]:
# Apply a custom function to each group
def normalize(series: pd.Series) -> pd.Series:
    series1 = (series - series.mean()) / series.std() 
    return series1

df['ni_Norm'] = df.groupby('gvkey')['ni'].transform(normalize)

In [ ]:
df['at_w_by_year'] = df.groupby('fyear')['at'].transform(lambda x: winsor2(x, p=0.05))

In [ ]:
df['at_w1_by_year'] = df.groupby('fyear')['at'].transform(winsorize)

#### 4.4. Apply: `.apply()`
The apply() method lets you apply a custom function to each group. This is useful for more complex operations that require a custom aggregation or transformation.

It can return a custom object or structure.

In [ ]:
# Apply a custom function to each group
def some_random_function(x: pd.DataFrame) -> pd.DataFrame:
    x['new_var'] = (x['ni'].max() - x['at'].min()) / (x['fyear'].count() + x['ceq'].min())
    return x

df_apply1 = df.groupby('gvkey', as_index=False).apply(some_random_function).reset_index(drop=True)
df_apply1.shape

In [ ]:
# Apply a custom function to each group
def some_random_function(x: pd.DataFrame) -> pd.DataFrame:
    y = pd.DataFrame({'gvkey': [x['gvkey'].iloc[0]], 
                      'new_var': [(x['ni'].max() - x['at'].min()) / (x['fyear'].count() + x['ceq'].min())]})
    return y

df_apply2 = df.groupby('gvkey', as_index=False).apply(some_random_function).reset_index(drop=True)
df_apply2.shape

#### 4.5. Filtering: `.filter()`
Sometimes you might want to filter the data based on the properties of the group.

For example, you might keep all the firms whose total asset is always larger than 100m.

In [ ]:
df_filtered = df.groupby('gvkey').filter(lambda x: x['at'].min() > 100)

#### 4.6. Shift: `.shift()`

In [ ]:
# Generate a lag or lead variable
df.sort_values(by=['gvkey','fyear'], inplace=True)
df['at_lag'] = df.groupby('gvkey')['at'].shift(1)
df['at_lead'] = df.groupby('gvkey')['at'].shift(-1, fill_value = df['at'].mean()) #Fill missing value with a scalar. Rarely used.

This is not a good practice as it cannot tell if the previous observation is indeed for the previous year.

You should add:

In [ ]:
df['at_lag'] = df['at_lag'].where(df.groupby('gvkey')['fyear'].diff()==1, np.nan)

### 5. Append and Merging

#### 5.1. Append using `pd.concat()`

In [ ]:
y1 = df.loc[1:1000]
z1 = df.loc[2001:2005]
stacked = pd.concat([y1,z1], axis=0, ignore_index=True)
print(len(stacked))

#### 5.2. Merge using `pd.merge()`
Basic:
```py
pd.merge(left_Dataframe, right_Dataframe, how='inner', on=[matching variables] {OR left_on= , right_on= })
```

In [ ]:
df1 = df.loc[1:75, ['gvkey','fyear','at','ceq']]
df2 = df.loc[51:150, ['gvkey','fyear','sale','ni']]

##### 5.2.1. One-to-one matching

In [ ]:
df_inner = pd.merge(df1,df2, on=['gvkey','fyear'], how='inner') #'inner' is default
df_left = pd.merge(df1,df2, on=['gvkey','fyear'], how='left')
df_right = pd.merge(df1,df2, on=['gvkey','fyear'], how='right')

In [ ]:
print(f'''
    Original Observations:
    df1: {len(df1)},
    df2: {len(df2)}
    Final Observations:
    Inner: {len(df_inner)}, Left: {len(df_left)}, Right: {len(df_right)}
    ''')

In [ ]:
# left_on and right_on are used when the column names are different
df1 = df.loc[1:75, ['gvkey','fyear','at','ceq']]
df2 = df.loc[51:150, ['gvkey','fyear','sale','ni']].rename(columns={'gvkey':'Not gvkey','fyear':'Not fyear'})

df_inner = pd.merge(df1,df2, left_on=['gvkey','fyear'], right_on=['Not gvkey','Not fyear']).drop(columns=['Not gvkey','Not fyear'])

In [ ]:
# A more complex example: Merge lag variable for a specific gvkey
df['fyear_lag'] = df['fyear'] + 1
pd.merge(df[['gvkey','fyear','at']], \
        df.loc[:, ['gvkey','fyear_lag','at']].rename(columns={'at': 'Lag_at'}), \
        how='left', \
        left_on=['gvkey','fyear'], \
        right_on=['gvkey','fyear_lag'])\
        .drop(columns='fyear_lag')
# This is a better practice than using `.shift()` as it ensures that the previous observation is indeed for the previous year

In [ ]:
# Same as:
df.merge(df[['gvkey','fyear_lag','at']].rename(columns={'at': 'Lag_at'}), how='left', left_on=['gvkey','fyear'], right_on=['gvkey','fyear_lag'])

##### 5.2.2. One-to-many matching

In [ ]:
Ever_RnD = (df.groupby('gvkey', as_index=False)['xrd']
        .apply(lambda x: x.notnull().any().astype(int))
        .rename(columns={'xrd':'Ever_RnD'}))

In [ ]:
df1 = df.merge(Ever_RnD, on='gvkey')

In [ ]:
# The process above is equivalent to:
df['Ever_RnD'] = df.groupby('gvkey', as_index=False)['xrd'].transform(lambda x: x.notnull().any().astype(int))

##### 5.2.3. Many-to-many merge

CCM: merge permno to Compustat data

In [ ]:
from datetime import date

In [ ]:
ccm = pd.read_csv('ccm.csv', parse_dates=['LINKDT','LINKENDDT'])

In [ ]:
ccm = ccm[ccm.LINKTYPE.isin(['LU','LC', 'LS'])].rename(columns={'LPERMNO':'permno'})

In [ ]:
#Cannot parse `LINKENDDT` because it contains missing values (E)
ccm['LINKENDDT'] = ccm['LINKENDDT'].apply(lambda x: pd.to_datetime(x, format='%Y%m%d') if len(x)==8 else pd.to_datetime(date.today()))

In [ ]:
df1 = df.merge(ccm[['gvkey','permno','LINKDT', 'LINKENDDT']], on='gvkey', how='inner').query('LINKDT <= datadate <= LINKENDDT')

##### 5.2.4. Merge using SQLite

In [ ]:
import sqlite3

In [ ]:
conn = sqlite3.connect(':memory:')
df.to_sql('comp',conn, index=False)
ccm.to_sql('link', conn, index=False)

In [ ]:
query = '''
     SELECT a.*, b.permno
     FROM comp AS a INNER JOIN link AS b
     ON a.gvkey = b.gvkey
     AND a.datadate BETWEEN b.LINKDT AND b.LINKENDDT
    '''

In [ ]:
df2 = pd.read_sql_query(query, conn)